# ⚡ Electricity Price Forecaster
### XGBoost + LSTM with Model Comparison

In [ ]:
# 📦 Install Dependencies
import subprocess, sys
for pkg in ['ipywidgets', 'pandas', 'numpy', 'plotly', 'xgboost', 'scikit-learn', 'tensorflow', 'openpyxl']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('✅ Ready!')

In [ ]:
# 🚀 Main Application

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, FileLink
from datetime import timedelta, datetime
import os
from xgboost import XGBRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ============== CONFIG ==============
DEFAULT_PATH = r"E:\DARSHIT\Enery\Downloads\DAP\Merged\ML XGboost\Merged_Hourly_DayAhead_2020_2025.xlsx"

# ============== STYLES ==============
display(HTML("""
<style>
.title-box { text-align:center; padding:20px; background:linear-gradient(135deg,#0f0c29,#302b63,#24243e); border-radius:12px; margin-bottom:15px; }
.title-box h1 { color:#f8f8f8; margin:0; font-size:28px; }
.title-box p { color:#aaa; margin:5px 0 0; }
.section { background:linear-gradient(135deg,#11998e,#38ef7d); color:white; padding:10px 15px; border-radius:8px; margin:15px 0 10px; font-weight:600; }
.info { background:#e3f2fd; border-left:4px solid #2196f3; padding:10px 12px; margin:8px 0; border-radius:0 6px 6px 0; }
.success { background:#e8f5e9; border-left:4px solid #4caf50; padding:12px; margin:8px 0; border-radius:0 6px 6px 0; }
.error { background:#ffebee; border-left:4px solid #f44336; padding:12px; margin:8px 0; border-radius:0 6px 6px 0; }
.metric { background:linear-gradient(135deg,#667eea,#764ba2); color:white; padding:12px; border-radius:8px; text-align:center; margin:4px; min-width:100px; display:inline-block; }
.metric h3 { margin:0; font-size:20px; }
.metric p { margin:3px 0 0; font-size:11px; opacity:0.9; }
.compare { background:#fff3e0; border-left:4px solid #ff9800; padding:12px; margin:10px 0; border-radius:0 6px 6px 0; }
</style>
"""))

# ============== DATA ==============
STATE = {'df': None}

def load_data(path):
    if not os.path.exists(path):
        return None, f"File not found: {path}"
    try:
        df = pd.read_excel(path) if path.endswith(('.xlsx','.xls')) else pd.read_csv(path)
        
        # Find columns
        ts_col = next((c for c in df.columns if any(x in c.lower() for x in ['time','date','timestamp'])), df.columns[0])
        price_col = next((c for c in df.columns if any(x in c.lower() for x in ['price','eur','mwh','value']) and c != ts_col), None)
        if not price_col:
            price_col = next((c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != ts_col), None)
        
        # Parse timestamps (handle CET/CEST A/B markers)
        ts = df[ts_col].astype(str).str.replace(r"(\d{4}-\d{2}-\d{2} )(\d{1,2})[AB]:", r"\g<1>\g<2>:", regex=True)
        df['timestamp'] = pd.to_datetime(ts, errors='coerce')
        df['price'] = pd.to_numeric(df[price_col], errors='coerce')
        
        df = df.dropna(subset=['timestamp','price']).set_index('timestamp')[['price']]
        df = df.groupby(df.index).mean().resample('1h').mean().interpolate(limit=3).dropna()
        return df, None
    except Exception as e:
        return None, str(e)

def create_xgb_features(df):
    d = df.copy()
    d['hour'], d['dow'], d['month'] = d.index.hour, d.index.dayofweek, d.index.month
    d['is_weekend'] = (d['dow'] >= 5).astype(int)
    d['hour_sin'] = np.sin(2*np.pi*d['hour']/24)
    d['hour_cos'] = np.cos(2*np.pi*d['hour']/24)
    for lag in [1,2,3,24,48,168]:
        d[f'lag_{lag}'] = d['price'].shift(lag)
    d['roll_24'] = d['price'].shift(1).rolling(24).mean()
    d['roll_168'] = d['price'].shift(1).rolling(168).mean()
    return d.dropna()

# ============== WIDGETS ==============
title = widgets.HTML("<div class='title-box'><h1>⚡ Price Forecaster</h1><p>XGBoost + LSTM • Model Comparison</p></div>")

file_input = widgets.Text(value=DEFAULT_PATH, description='Path:', layout=widgets.Layout(width='95%'), style={'description_width':'50px'})
load_btn = widgets.Button(description=' Load', icon='folder-open', button_style='info', layout=widgets.Layout(width='100px'))
load_out = widgets.HTML()
data_info = widgets.HTML()

from_dt = widgets.DatePicker(description='From:', value=datetime.now().date()-timedelta(days=180), style={'description_width':'50px'})
to_dt = widgets.DatePicker(description='To:', value=datetime.now().date(), style={'description_width':'50px'})
horizon = widgets.IntSlider(value=3, min=1, max=7, description='Days:', style={'description_width':'50px'}, layout=widgets.Layout(width='300px'))

model_choice = widgets.RadioButtons(options=['XGBoost Only', 'LSTM Only', 'Both (Compare)'], value='Both (Compare)', description='Model:')

run_btn = widgets.Button(description=' Run Forecast', icon='play', button_style='success', layout=widgets.Layout(width='180px', height='45px'))
output = widgets.Output()

def on_load(b):
    load_out.value = "<div class='info'>⏳ Loading...</div>"
    df, err = load_data(file_input.value.strip())
    if err:
        load_out.value = f"<div class='error'>❌ {err}</div>"
        STATE['df'] = None
        data_info.value = ""
    else:
        STATE['df'] = df
        from_dt.value = max(df.index.min().date(), df.index.max().date() - timedelta(days=365))
        to_dt.value = df.index.max().date()
        load_out.value = "<div class='success'>✅ Loaded!</div>"
        data_info.value = f"<div class='info'><b>{len(df):,}</b> records | {df.index.min().strftime('%Y-%m-%d')} → {df.index.max().strftime('%Y-%m-%d')} | Avg: €{df['price'].mean():.1f}/MWh</div>"

load_btn.on_click(on_load)

# ============== FORECASTING ==============
def run_xgboost(df, horizon_h):
    """XGBoost forecast with proper validation."""
    df_feat = create_xgb_features(df)
    feat_cols = [c for c in df_feat.columns if c != 'price']
    
    # Time-based split (no shuffle!)
    split = int(len(df_feat) * 0.85)
    X_tr, X_te = df_feat[feat_cols].iloc[:split], df_feat[feat_cols].iloc[split:]
    y_tr, y_te = df_feat['price'].iloc[:split], df_feat['price'].iloc[split:]
    
    model = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, n_jobs=-1, random_state=42)
    model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
    
    # Metrics
    y_pred = model.predict(X_te)
    metrics = {
        'rmse': np.sqrt(mean_squared_error(y_te, y_pred)),
        'mae': mean_absolute_error(y_te, y_pred),
        'mape': mean_absolute_percentage_error(y_te, y_pred) * 100
    }
    
    # Forecast
    prices = df['price'].tolist()
    last_ts = df.index[-1]
    forecasts = []
    
    for i in range(horizon_h):
        ts = last_ts + timedelta(hours=i+1)
        feat = {
            'hour': ts.hour, 'dow': ts.weekday(), 'month': ts.month,
            'is_weekend': 1 if ts.weekday()>=5 else 0,
            'hour_sin': np.sin(2*np.pi*ts.hour/24),
            'hour_cos': np.cos(2*np.pi*ts.hour/24),
        }
        for lag in [1,2,3,24,48,168]:
            feat[f'lag_{lag}'] = prices[-lag] if len(prices)>=lag else prices[0]
        feat['roll_24'] = np.mean(prices[-24:]) if len(prices)>=24 else np.mean(prices)
        feat['roll_168'] = np.mean(prices[-168:]) if len(prices)>=168 else np.mean(prices)
        
        pred = model.predict(pd.DataFrame([feat])[feat_cols])[0]
        forecasts.append({'timestamp': ts, 'price': pred})
        prices.append(pred)
    
    return pd.DataFrame(forecasts), metrics

def run_lstm(df, horizon_h):
    """LSTM forecast with FIXED validation (no data leakage)."""
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Input
    from tensorflow.keras.callbacks import EarlyStopping
    
    # Add time features to help LSTM
    data = df.copy()
    data['hour_sin'] = np.sin(2*np.pi*data.index.hour/24)
    data['hour_cos'] = np.cos(2*np.pi*data.index.hour/24)
    data['dow_sin'] = np.sin(2*np.pi*data.index.dayofweek/7)
    
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(data.values)
    
    window = 168  # 1 week lookback
    X, y = [], []
    for i in range(window, len(scaled)):
        X.append(scaled[i-window:i])
        y.append(scaled[i, 0])  # Predict next price only
    X, y = np.array(X), np.array(y)
    
    # FIXED: Time-based split, NO shuffle
    split = int(len(X) * 0.85)
    X_tr, X_te = X[:split], X[split:]
    y_tr, y_te = y[:split], y[split:]
    
    model = Sequential([
        Input(shape=(window, X.shape[2])),
        LSTM(64, activation='tanh'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    
    # FIXED: shuffle=False for time series!
    early_stop = EarlyStopping(patience=5, restore_best_weights=True)
    model.fit(X_tr, y_tr, epochs=50, batch_size=32, validation_data=(X_te, y_te),
              callbacks=[early_stop], verbose=0, shuffle=False)
    
    # Metrics
    y_pred_sc = model.predict(X_te, verbose=0).flatten()
    # Inverse transform just the price column
    dummy = np.zeros((len(y_pred_sc), data.shape[1]))
    dummy[:, 0] = y_pred_sc
    y_pred = scaler.inverse_transform(dummy)[:, 0]
    dummy[:, 0] = y_te
    y_true = scaler.inverse_transform(dummy)[:, 0]
    
    metrics = {
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae': mean_absolute_error(y_true, y_pred),
        'mape': mean_absolute_percentage_error(y_true, y_pred) * 100
    }
    
    # Recursive forecast
    last_seq = scaled[-window:].copy()
    last_ts = df.index[-1]
    forecasts = []
    
    for i in range(horizon_h):
        ts = last_ts + timedelta(hours=i+1)
        pred_sc = model.predict(last_seq.reshape(1, window, -1), verbose=0)[0, 0]
        
        # Inverse transform
        dummy_row = np.zeros((1, data.shape[1]))
        dummy_row[0, 0] = pred_sc
        pred = scaler.inverse_transform(dummy_row)[0, 0]
        forecasts.append({'timestamp': ts, 'price': pred})
        
        # Update sequence with new prediction + time features
        new_row = np.array([[pred_sc, 
                            np.sin(2*np.pi*ts.hour/24),
                            np.cos(2*np.pi*ts.hour/24),
                            np.sin(2*np.pi*ts.weekday()/7)]])
        # Scale the time features properly
        new_row[0, 1:] = (new_row[0, 1:] - scaler.data_min_[1:]) / (scaler.data_max_[1:] - scaler.data_min_[1:] + 1e-10)
        last_seq = np.vstack([last_seq[1:], new_row])
    
    return pd.DataFrame(forecasts), metrics

def on_run(b):
    with output:
        clear_output()
        
        if STATE['df'] is None:
            display(HTML("<div class='error'>❌ Load data first!</div>"))
            return
        
        # Filter data
        df = STATE['df'][(STATE['df'].index.date >= from_dt.value) & (STATE['df'].index.date <= to_dt.value)].copy()
        if len(df) < 500:
            display(HTML(f"<div class='error'>❌ Need 500+ records, got {len(df)}. Expand date range.</div>"))
            return
        
        horizon_h = horizon.value * 24
        choice = model_choice.value
        results = {}
        
        # Progress
        prog = widgets.IntProgress(value=0, max=100, layout=widgets.Layout(width='100%'))
        status = widgets.HTML('<b>⏳ Running...</b>')
        display(widgets.VBox([status, prog]))
        
        try:
            if 'XGBoost' in choice or 'Both' in choice:
                status.value = '<b>⏳ Training XGBoost...</b>'
                prog.value = 20
                results['XGBoost'] = run_xgboost(df, horizon_h)
                prog.value = 50
            
            if 'LSTM' in choice or 'Both' in choice:
                status.value = '<b>⏳ Training LSTM (may take a minute)...</b>'
                prog.value = 60
                results['LSTM'] = run_lstm(df, horizon_h)
                prog.value = 90
            
            prog.value = 100
            clear_output()
            
            # ========== DISPLAY RESULTS ==========
            
            # Metrics comparison
            display(HTML("<div class='section'>📊 Model Performance</div>"))
            
            metrics_html = "<div style='display:flex;flex-wrap:wrap;gap:10px;'>"
            for name, (fc, met) in results.items():
                color = '#667eea' if name == 'XGBoost' else '#f093fb'
                metrics_html += f"""
                <div style='border:2px solid {color};border-radius:10px;padding:10px;min-width:200px;'>
                    <h4 style='margin:0;color:{color};'>{name}</h4>
                    <p style='margin:5px 0;'>RMSE: <b>€{met['rmse']:.2f}</b></p>
                    <p style='margin:5px 0;'>MAE: <b>€{met['mae']:.2f}</b></p>
                    <p style='margin:5px 0;'>MAPE: <b>{met['mape']:.1f}%</b></p>
                </div>
                """
            metrics_html += "</div>"
            display(HTML(metrics_html))
            
            # Winner announcement for comparison
            if len(results) == 2:
                xgb_mape = results['XGBoost'][1]['mape']
                lstm_mape = results['LSTM'][1]['mape']
                winner = 'XGBoost' if xgb_mape < lstm_mape else 'LSTM'
                diff = abs(xgb_mape - lstm_mape)
                display(HTML(f"<div class='compare'>🏆 <b>{winner}</b> wins by {diff:.1f}% MAPE</div>"))
            
            # Daily Summary
            display(HTML("<div class='section'>📅 Daily Summary</div>"))
            
            # Use first model's forecast for summary (or average if both)
            if len(results) == 2:
                fc1 = results['XGBoost'][0].copy()
                fc2 = results['LSTM'][0].copy()
                fc1['price_lstm'] = fc2['price']
                fc1['price_avg'] = (fc1['price'] + fc1['price_lstm']) / 2
                main_fc = fc1
                price_col = 'price_avg'
            else:
                main_fc = list(results.values())[0][0]
                price_col = 'price'
            
            main_fc['date'] = main_fc['timestamp'].dt.date
            main_fc['hour'] = main_fc['timestamp'].dt.hour
            
            daily = []
            for date in main_fc['date'].unique():
                day = main_fc[main_fc['date'] == date]
                low_idx = day[price_col].idxmin()
                high_idx = day[price_col].idxmax()
                daily.append({
                    'Date': pd.Timestamp(date).strftime('%a %d %b'),
                    'Low Hour': f"{int(day.loc[low_idx, 'hour']):02d}:00",
                    'Low €': day.loc[low_idx, price_col],
                    'High Hour': f"{int(day.loc[high_idx, 'hour']):02d}:00",
                    'High €': day.loc[high_idx, price_col],
                    'Spread': day[price_col].max() - day[price_col].min()
                })
            
            daily_df = pd.DataFrame(daily)
            display(daily_df.style.format({'Low €': '€{:.2f}', 'High €': '€{:.2f}', 'Spread': '€{:.2f}'}
                                         ).background_gradient(subset=['Spread'], cmap='YlOrRd'))
            
            # Chart
            display(HTML("<div class='section'>📈 Forecast Chart</div>"))
            
            fig = go.Figure()
            
            # Historical
            hist = df[df.index >= df.index[-1] - timedelta(days=5)]
            fig.add_trace(go.Scatter(x=hist.index, y=hist['price'], name='Historical',
                                     line=dict(color='#2196f3', width=2)))
            
            # Forecasts
            colors = {'XGBoost': '#4caf50', 'LSTM': '#ff5722'}
            for name, (fc, _) in results.items():
                fig.add_trace(go.Scatter(x=fc['timestamp'], y=fc['price'], name=name,
                                         line=dict(color=colors.get(name, '#999'), width=2, dash='dot')))
            
            fig.update_layout(title='Price Forecast Comparison', template='plotly_white', height=450,
                             xaxis_title='Time', yaxis_title='€/MWh', hovermode='x unified',
                             legend=dict(orientation='h', y=1.02))
            fig.show()
            
            # Heatmap (if both models)
            if len(results) == 2:
                display(HTML("<div class='section'>🔥 Model Difference Heatmap</div>"))
                
                fc1 = results['XGBoost'][0].copy()
                fc1['lstm'] = results['LSTM'][0]['price']
                fc1['diff'] = fc1['price'] - fc1['lstm']
                fc1['date'] = fc1['timestamp'].dt.strftime('%a %d')
                fc1['hour'] = fc1['timestamp'].dt.hour
                
                pivot = fc1.pivot_table(index='date', columns='hour', values='diff')
                
                fig2 = go.Figure(go.Heatmap(
                    z=pivot.values, x=[f'{h:02d}:00' for h in range(24)], y=pivot.index,
                    colorscale='RdBu', zmid=0, colorbar=dict(title='XGB - LSTM')
                ))
                fig2.update_layout(title='XGBoost vs LSTM Difference (€/MWh)', template='plotly_white',
                                   height=200 + len(pivot)*30, xaxis_title='Hour', yaxis_title='Date')
                fig2.show()
            
            # Downloads
            display(HTML("<div class='section'>💾 Download</div>"))
            for name, (fc, _) in results.items():
                fname = f"{name.lower()}_forecast_{datetime.now().strftime('%Y%m%d')}.csv"
                fc.to_csv(fname, index=False)
                display(FileLink(fname, result_html_prefix=f'{name}: '))
        
        except Exception as e:
            clear_output()
            display(HTML(f"<div class='error'>❌ Error: {str(e)}</div>"))

run_btn.on_click(on_run)

# ============== LAYOUT ==============
ui = widgets.VBox([
    title,
    widgets.HTML("<div class='section'>📁 Data</div>"),
    file_input, load_btn, load_out, data_info,
    widgets.HTML("<div class='section'>⚙️ Settings</div>"),
    widgets.HBox([from_dt, to_dt]), horizon, model_choice,
    widgets.HTML("<br>"),
    widgets.HBox([run_btn], layout=widgets.Layout(justify_content='center')),
    widgets.HTML("<br>"),
    output
], layout=widgets.Layout(max_width='900px', padding='15px'))

display(ui)

---
### 📝 Notes

**XGBoost** - Fast, interpretable, uses hand-crafted features (lags, rolling stats, time)

**LSTM** - Deep learning, learns patterns automatically, but slower and needs more data

**Key Fixes in LSTM:**
- ✅ Time-based split (no shuffle) - prevents data leakage
- ✅ Added time features (hour, day) - helps capture patterns
- ✅ Early stopping - prevents overfitting
- ✅ Proper metrics calculation